# CMD commands

In [1]:
%pip install -r ../requirements.txt

  Obtaining dependency information for torch>=2.0.0 from https://files.pythonhosted.org/packages/6e/01/624c4324ca01f66ae4c7cd1b74eb16fb52596dce66dbe51eff95ef9e7a4c/torch-2.10.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torch-2.10.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Obtaining dependency information for torchvision>=0.15.0 from https://files.pythonhosted.org/packages/ad/16/8f650c2e288977cf0f8f85184b90ee56ed170a4919347fc74ee99286ed6f/torchvision-0.25.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torchvision-0.25.0-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Obtaining dependency information for facenet-pytorch>=2.5.2 from https://files.pythonhosted.org/packages/ed/2e/2d56386bc2f834cdc683743903852cf1428b4e5ee119f16cf808b589d3cd/facenet_pytorch-2.6.0-py3-none-any.whl.metadata
  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
  Obtaining dependency information for opencv-python>=4.8.0 from https://files.pythonhosted.org/packages/e9/a5/1be1516390


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Imports

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from facenet_pytorch import MTCNN
from PIL import Image
import torch


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Paths

In [2]:
# === INPUT PATHS ===
CSV_PATH = "../Data/MELD.Raw/train/train_sent_emo.csv"
VIDEO_FOLDER = "../Data/MELD.Raw/train/train_splits"

# === OUTPUT PATHS ===
OUTPUT_ROOT = "../data/processed_meld"
IMAGE_OUTPUT = os.path.join(OUTPUT_ROOT, "images", "train")
METADATA_OUTPUT = os.path.join(OUTPUT_ROOT, "metadata_train.csv")

os.makedirs(IMAGE_OUTPUT, exist_ok=True)


# Initialize Face Detector

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(
    keep_all=True,
    device=device
)


# Helper Functions

In [5]:
def select_central_face(boxes, frame_width, frame_height):
    """
    Select face closest to image center.
    boxes: list of [x1, y1, x2, y2]
    """
    if boxes is None:
        return None
    
    center_x = frame_width / 2
    center_y = frame_height / 2
    
    min_dist = float("inf")
    best_box = None
    
    for box in boxes:
        x1, y1, x2, y2 = box
        face_center_x = (x1 + x2) / 2
        face_center_y = (y1 + y2) / 2
        
        dist = (face_center_x - center_x)**2 + (face_center_y - center_y)**2
        
        if dist < min_dist:
            min_dist = dist
            best_box = box
    
    return best_box


# Main Processing Loop

In [6]:
df = pd.read_csv(CSV_PATH)

metadata_rows = []

frame_positions = [0.1, 0.3, 0.5, 0.7, 0.9]

for idx, row in tqdm(df.iterrows(), total=len(df)):
    
    dialogue_id = row["Dialogue_ID"]
    utterance_id = row["Utterance_ID"]
    emotion = row["Emotion"]
    
    video_name = f"dia{dialogue_id}_utt{utterance_id}.mp4"
    video_path = os.path.join(VIDEO_FOLDER, video_name)
    
    if not os.path.exists(video_path):
        continue
    
    # Create emotion folder
    emotion_folder = os.path.join(IMAGE_OUTPUT, emotion)
    os.makedirs(emotion_folder, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        continue
    
    for i, rel_pos in enumerate(frame_positions):
        
        frame_number = int(total_frames * rel_pos)
        frame_number = min(frame_number, total_frames - 1)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        
        if not ret:
            continue
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        
        boxes, _ = mtcnn.detect(pil_image)
        
        if boxes is None:
            continue
        
        h, w, _ = frame.shape
        best_box = select_central_face(boxes, w, h)
        
        if best_box is None:
            continue
        
        x1, y1, x2, y2 = [int(b) for b in best_box]
        face_crop = frame[y1:y2, x1:x2]
        
        if face_crop.size == 0:
            continue
        
        image_name = f"d{dialogue_id}_u{utterance_id}_f{i}.jpg"
        image_path = os.path.join(emotion_folder, image_name)
        
        cv2.imwrite(image_path, face_crop)
        
        metadata_rows.append({
            "image_path": image_path,
            "emotion": emotion,
            "dialogue_id": dialogue_id,
            "utterance_id": utterance_id,
            "frame_index": i
        })
    
    cap.release()

  0%|          | 19/9989 [00:14<2:04:35,  1.33it/s]


KeyboardInterrupt: 

# Save Metadata

In [ ]:
metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(METADATA_OUTPUT, index=False)

print("Done. Total images:", len(metadata_df))
